<div style="background:linear-gradient(135deg,#1e3a8a 0%,#0c1a3d 100%);border-radius:16px;padding:30px 34px;color:#eff6ff;margin-bottom:6px;">
<div style="font-size:12.5px;letter-spacing:3px;text-transform:uppercase;opacity:.65;font-weight:600;">News Article Topic Classification</div>
<div style="font-size:30px;font-weight:800;margin:6px 0 10px;">03 · Modeling</div>
<div style="font-size:14.5px;opacity:.92;max-width:640px;line-height:1.55;">Trains every allowed model — GPU-accelerated where possible — and combines them with a final soft-voting ensemble.</div>
</div>

<div style="margin:12px 0 6px;">
<span style="display:inline-block;background:#e2e8f0;color:#64748b;padding:4px 12px;border-radius:20px;font-size:11.5px;margin-right:5px;">01 Load Data</span><span style="display:inline-block;background:#e2e8f0;color:#64748b;padding:4px 12px;border-radius:20px;font-size:11.5px;margin-right:5px;">02 Preprocessing</span><span style="display:inline-block;background:#1e3a8a;color:#fff;padding:4px 12px;border-radius:20px;font-size:11.5px;margin-right:5px;font-weight:600;">03 Modeling</span><span style="display:inline-block;background:#e2e8f0;color:#64748b;padding:4px 12px;border-radius:20px;font-size:11.5px;margin-right:5px;">04 Evaluation & Testing</span><span style="display:inline-block;background:#e2e8f0;color:#64748b;padding:4px 12px;border-radius:20px;font-size:11.5px;margin-right:5px;">05 Visualization</span>
</div>

<div style="margin:8px 0 14px;">
<span style="display:inline-block;background:#eff6ff;border:1px solid #93c5fd;border-radius:8px;padding:7px 14px;font-size:13px;margin-right:8px;">
<b style="color:#1e3a8a;">⬅ Requires</b>&nbsp; <code>artifacts/02_preprocessing.pkl</code> — run <code>02_preprocessing.ipynb</code> once beforehand
</span><span style="display:inline-block;background:#eff6ff;border:1px solid #93c5fd;border-radius:8px;padding:7px 14px;font-size:13px;margin-right:8px;">
<b style="color:#1e3a8a;">➡ Produces</b>&nbsp; <code>artifacts/03_modeling.pkl</code> — read by <code>04_evaluation_testing.ipynb</code> and <code>05_visualization.ipynb</code>
</span>
</div>

## <span style="color:#fff;">Load Artifacts</span>
<div style="height:3px;width:48px;background:#2563eb;border-radius:2px;margin:2px 0 10px;"></div>



In [7]:
import numpy as np
import pandas as pd
import os
import pickle
import time

from google.colab import drive
drive.mount('/content/drive')

BASE_PATH = '/content/drive/MyDrive/Projects/DSL/News Classification/'
ARTIFACT_DIR = BASE_PATH + 'artifacts/'

with open(ARTIFACT_DIR + '02_preprocessing.pkl', 'rb') as f:
    _art02 = pickle.load(f)

X_Train_final          = _art02['X_Train_final']
X_Validation_final     = _art02['X_Validation_final']
evaluation_final       = _art02['evaluation_final']
X_Train_text_svd       = _art02['X_Train_text_svd']
X_Validation_text_svd  = _art02['X_Validation_text_svd']
evaluation_text_svd    = _art02['evaluation_text_svd']
Y_Train_aligned        = _art02['Y_Train_aligned']
Y_Validation_aligned   = _art02['Y_Validation_aligned']
X_Train                = _art02['X_Train']
evaluation              = _art02['evaluation']
submission              = _art02['submission']
SEED                    = _art02['SEED']

print(f"✓ Loaded preprocessing artifacts from {ARTIFACT_DIR}02_preprocessing.pkl")
print(f"  X_Train_final {X_Train_final.shape}, X_Train_text_svd {X_Train_text_svd.shape}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ Loaded preprocessing artifacts from /content/drive/MyDrive/Projects/DSL/News Classification/artifacts/02_preprocessing.pkl
  X_Train_final (34869, 1095), X_Train_text_svd (34869, 300)


## <span style="color:#fff;">GPU Backend</span>
<div style="height:3px;width:48px;background:#2563eb;border-radius:2px;margin:2px 0 10px;"></div>

Detects cuML (RAPIDS) for GPU-accelerated LogReg/SGD/SVM/KNN/RandomForest/NaiveBayes, and a CUDA-capable PyTorch for the MLP. Falls back to CPU automatically if no GPU is attached.

In [8]:
import warnings
warnings.filterwarnings('ignore')

try:
    import cuml
    from cuml.linear_model import LogisticRegression as _LR_gpu
    from cuml.neighbors import KNeighborsClassifier as _KNN_gpu
    from cuml.ensemble import RandomForestClassifier as _RF_gpu
    from cuml.naive_bayes import GaussianNB as _GNB_gpu
    _GPU_BACKEND = True
    print('✓ GPU backend: cuML available — LogReg, KNN, RandomForest, GaussianNB run on GPU')
    print("  (SGDClassifier and both SVMs stay on CPU — cuML's MBSGDClassifier/SVC are binary-only, this is a 7-class problem)")
except ImportError:
    _GPU_BACKEND = False
    print('ℹ cuML not available — all classical models fall back to CPU (sklearn)')

try:
    import torch
    _TORCH_DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f'✓ PyTorch device for MLP: {_TORCH_DEVICE}')
except ImportError:
    torch = None
    _TORCH_DEVICE = 'cpu'
    print('ℹ PyTorch not available — MLP step will be skipped')

✓ GPU backend: cuML available — LogReg, SGD, SVM(linear/RBF), KNN, RandomForest, GaussianNB run on GPU
✓ PyTorch device for MLP: cuda


## <span style="color:#fff;">Hyperparameter Tuning: Top Linear Models</span>
<div style="height:3px;width:48px;background:#2563eb;border-radius:2px;margin:2px 0 10px;"></div>

A light `RandomizedSearchCV` (3-fold, small grids) on the three models that led the first leaderboard run — Logistic Regression, LinearSVC, SGDClassifier — since they were already clustered tightly and tuning is the cheapest way left to squeeze more out of them.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import make_scorer, f1_score as _f1

_macro_f1_scorer = make_scorer(_f1, average='macro')

_search_space = {
    'Logistic Regression': (LogisticRegression(class_weight='balanced', max_iter=1000, n_jobs=-1),
                             {'C': [0.1, 0.3, 1.0, 3.0, 10.0]}),
    'LinearSVC':           (LinearSVC(class_weight='balanced', max_iter=3000, dual=True, random_state=SEED),
                             {'C': [0.1, 0.3, 1.0, 3.0]}),
    'SGDClassifier':       (SGDClassifier(loss='log_loss', max_iter=2000, n_jobs=-1, random_state=SEED),
                             {'alpha': [1e-6, 1e-5, 1e-4], 'penalty': ['l2', 'elasticnet']}),
}

tuned_params = {}
for _name, (_est, _grid) in _search_space.items():
    _t0 = time.time()
    _search = RandomizedSearchCV(_est, _grid, n_iter=6, scoring=_macro_f1_scorer, cv=3,
                                  random_state=SEED, n_jobs=-1)
    _search.fit(X_Train_final, Y_Train_aligned)
    tuned_params[_name] = _search.best_params_
    print(f"  {_name:<22} best={_search.best_params_}  cv_macro_f1={_search.best_score_:.4f}  ({time.time()-_t0:.1f}s)")

print("\n✓ tuned_params ready — used below when the model zoo is built")

## <span style="color:#fff;">Train the Allowed Model Zoo</span>
<div style="height:3px;width:48px;background:#2563eb;border-radius:2px;margin:2px 0 10px;"></div>

Every model here is from the approved list only: Decision Tree, Random Forest, Extra Trees, Gradient Boosting (sklearn, CPU by design), Logistic Regression, SGDClassifier, LinearSVC(+Calibrated), RidgeClassifier(+Calibrated), Naive Bayes, KNN, SVM (RBF). GPU is used only for Random Forest, Logistic Regression, KNN, and Naive Bayes — cuML's <code>MBSGDClassifier</code> and <code>SVC</code> only support <b>binary</b> classification, so SGDClassifier and both SVMs stay on CPU for this 7-class problem, regardless of GPU availability. Naive Bayes uses <b>GaussianNB</b>, not Multinomial/Complement — the combined feature matrix is dense and can be negative after TruncatedSVD, which those require non-negative input to avoid.

<b>Second-pass changes</b>, based on the first run's actual leaderboard: the tree models are now regularized (were badly overfit), Decision Tree/Extra Trees are calibrated, Gradient Boosting is cut down for speed (73 min → expected well under 15 min) and made more conservative in the same move, SVM (RBF) fits on a 15,000-row subsample instead of the full training set, and Logistic Regression/LinearSVC/SGDClassifier use the tuned hyperparameters from the cell above. The train/val gap is still measured on a fixed 3,000-row training subsample, not the full training set.

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression, SGDClassifier, RidgeClassifier
from sklearn.svm import LinearSVC, SVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split as _tts
from sklearn.metrics import f1_score, classification_report, confusion_matrix

CW = 'balanced'

_lr_kwargs = {'C': 1.0}
_lr_kwargs.update(tuned_params.get('Logistic Regression', {}))
_svc_kwargs = {'C': 1.0}
_svc_kwargs.update(tuned_params.get('LinearSVC', {}))
_sgd_kwargs = {'alpha': 1e-5}
_sgd_kwargs.update(tuned_params.get('SGDClassifier', {}))

# LinearSVC / RidgeClassifier used to be wrapped in CalibratedClassifierCV(cv=3),
# which doesn't calibrate a single full-data fit -- it internally refits the
# base estimator on 3 separate 2/3 folds and averages their calibrated
# probabilities, so .predict() was the argmax of 3 undertrained sub-models
# instead of the decision boundary of one model fit on all the data. That
# measurably cost Macro F1 vs. the first (uncalibrated) run. Fix: fit the base
# estimator once on a train slice, then calibrate with cv='prefit' on a small
# held-out slice -- one full-strength model, calibrated probabilities.
_cal_fit_idx, _cal_holdout_idx = _tts(
    np.arange(len(X_Train_final)), test_size=0.1, random_state=SEED, stratify=Y_Train_aligned)
PREFIT_CALIBRATED = {'LinearSVC', 'RidgeClassifier'}

# Model zoo: every entry restricted to the approved list. GPU (cuML) is
# used only where it actually supports multiclass: RandomForest, Logistic
# Regression, KNN, GaussianNB. cuML's MBSGDClassifier and SVC are
# binary-only, so SGDClassifier and both SVMs stay on CPU regardless of
# GPU availability -- this is a 7-class problem, not a limitation we can
# code around.
#
# Second-pass changes, based on the first run's actual leaderboard:
#   - Decision Tree / Random Forest / Extra Trees: max_depth 20 -> 12,
#     min_samples_leaf=5 -- the first run showed train/val gaps of
#     0.24-0.30 (e.g. Extra Trees 0.945 train vs 0.693 val), i.e. badly
#     overfit.
#   - Decision Tree / Extra Trees additionally wrapped in
#     CalibratedClassifierCV -- both are cheap (seconds), so calibrating
#     them costs little and makes their contribution to the final vote
#     more trustworthy. Gradient Boosting/Random Forest are NOT wrapped --
#     GB is already the most expensive model here (calibration would
#     ~3x that), and wrapping the GPU RandomForest risks cuML/sklearn
#     compatibility issues.
#   - Gradient Boosting: n_estimators 150->80, max_depth 3->2, subsample=0.5,
#     learning_rate raised to 0.15 to compensate for fewer/shallower trees.
#     The first run took 4366s (~73 min) for 0.714 val -- worse time/value
#     than free Logistic Regression at 0.723. This should cut that
#     drastically while also regularizing it (subsample < 1.0 is itself a
#     regularizer).
#   - Logistic Regression / LinearSVC / SGDClassifier: now built from
#     tuned_params (previous cell).
#   - LinearSVC / RidgeClassifier: fit once on a train slice, calibrated
#     with cv='prefit' on a held-out slice (see note above) instead of
#     CalibratedClassifierCV(cv=3).
model_defs = {
    'Decision Tree':      CalibratedClassifierCV(
                               DecisionTreeClassifier(class_weight=CW, max_depth=12, min_samples_leaf=5, random_state=SEED),
                               cv=3),
    'Random Forest':      (_RF_gpu(n_estimators=300, max_depth=12, random_state=SEED) if _GPU_BACKEND
                            else RandomForestClassifier(class_weight=CW, n_estimators=300, max_depth=12, min_samples_leaf=5, n_jobs=-1, random_state=SEED)),
    'Extra Trees':        CalibratedClassifierCV(
                               ExtraTreesClassifier(class_weight=CW, n_estimators=300, max_depth=12, min_samples_leaf=5, n_jobs=-1, random_state=SEED),
                               cv=3),
    'Gradient Boosting':  GradientBoostingClassifier(n_estimators=80, max_depth=2, subsample=0.5, learning_rate=0.15, random_state=SEED),
    'Logistic Regression': (_LR_gpu(max_iter=1000, **_lr_kwargs) if _GPU_BACKEND
                             else LogisticRegression(class_weight=CW, max_iter=1000, n_jobs=-1, **_lr_kwargs)),
    'SGDClassifier':      SGDClassifier(loss='log_loss', max_iter=2000, n_jobs=-1, random_state=SEED, **_sgd_kwargs),
    'LinearSVC':          LinearSVC(class_weight=CW, max_iter=3000, dual=True, random_state=SEED, **_svc_kwargs),
    'RidgeClassifier':    RidgeClassifier(class_weight=CW, random_state=SEED),
    'Naive Bayes':        (_GNB_gpu() if _GPU_BACKEND else GaussianNB()),
    'KNN':                (_KNN_gpu(n_neighbors=15) if _GPU_BACKEND else KNeighborsClassifier(n_neighbors=15, n_jobs=-1)),
    'SVM (RBF)':          SVC(class_weight=CW, kernel='rbf', probability=True, random_state=SEED),
}

# Kernel SVM cost scales roughly quadratically with sample count -- fit it
# on a fixed subsample to keep it tractable. It still predicts on the full
# validation set as normal.
SVM_SUBSAMPLE = min(15000, len(X_Train_final))
_rng_svm = np.random.RandomState(SEED)
_svm_idx = _rng_svm.choice(len(X_Train_final), size=SVM_SUBSAMPLE, replace=False)

results = {}
proba_val = {}

# The train/val gap is a diagnostic only (not used to pick the final model),
# so it's computed on a fixed random subsample of the training set -- full-
# training-set inference is genuinely expensive for KNN/SVM(RBF), whose
# cost scales with how many training points they compare against, and
# doubling that cost buys nothing here.
_rng = np.random.RandomState(SEED)
_diag_idx = _rng.choice(len(X_Train_final), size=min(3000, len(X_Train_final)), replace=False)
X_Train_diag = X_Train_final[_diag_idx]
Y_Train_diag = Y_Train_aligned[_diag_idx]

print("=" * 80)
print(f"{'Model':<22} {'Train F1':>10} {'Val F1':>10} {'Gap':>8} {'Time(s)':>9}")
print("=" * 80)

for name, model in model_defs.items():
    t0 = time.time()
    if name == 'SVM (RBF)':
        model.fit(X_Train_final[_svm_idx], Y_Train_aligned[_svm_idx])
    elif name in PREFIT_CALIBRATED:
        model.fit(X_Train_final[_cal_fit_idx], Y_Train_aligned[_cal_fit_idx])
        model = CalibratedClassifierCV(estimator=model, cv='prefit')
        model.fit(X_Train_final[_cal_holdout_idx], Y_Train_aligned[_cal_holdout_idx])
    else:
        model.fit(X_Train_final, Y_Train_aligned)
    elapsed = time.time() - t0

    p_train = model.predict(X_Train_diag)
    p_val   = model.predict(X_Validation_final)
    train_f1 = f1_score(Y_Train_diag, p_train, average='macro')
    val_f1   = f1_score(Y_Validation_aligned, p_val, average='macro')

    proba = np.asarray(model.predict_proba(X_Validation_final))
    proba_val[name] = proba

    results[name] = {'model': model, 'train_macro': train_f1, 'val_macro': val_f1, 'time': elapsed}
    print(f"{name:<22} {train_f1:>10.4f} {val_f1:>10.4f} {train_f1-val_f1:>8.4f} {elapsed:>9.1f}")

print("=" * 80)

## <span style="color:#fff;">MLP (TF-IDF only)</span>
<div style="height:3px;width:48px;background:#2563eb;border-radius:2px;margin:2px 0 10px;"></div>

Per spec, the MLP trains only on the TF-IDF-derived block (<code>X_Train_text_svd</code>), not the combined feature matrix — a small feed-forward network on GPU via PyTorch.

In [ ]:
# ── Simple MLP on TF-IDF (post chi2+SVD) — trained on GPU when available ──
if torch is not None:
    import torch.nn as nn
    import torch.optim as optim
    from torch.utils.data import TensorDataset, DataLoader

    torch.manual_seed(SEED)

    n_classes = len(np.unique(Y_Train_aligned))
    input_dim = X_Train_text_svd.shape[1]

    class NewsMLP(nn.Module):
        def __init__(self, input_dim, n_classes, hidden=(256, 128)):
            super().__init__()
            layers = []
            prev = input_dim
            for h in hidden:
                layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(0.3)]
                prev = h
            layers.append(nn.Linear(prev, n_classes))
            self.net = nn.Sequential(*layers)

        def forward(self, x):
            return self.net(x)

    mlp = NewsMLP(input_dim, n_classes).to(_TORCH_DEVICE)
    optimizer = optim.Adam(mlp.parameters(), lr=1e-3, weight_decay=1e-5)
    criterion = nn.CrossEntropyLoss()

    X_tr_t = torch.tensor(X_Train_text_svd, dtype=torch.float32)
    y_tr_t = torch.tensor(Y_Train_aligned, dtype=torch.long)
    loader = DataLoader(TensorDataset(X_tr_t, y_tr_t), batch_size=512, shuffle=True)

    X_val_t = torch.tensor(X_Validation_text_svd, dtype=torch.float32).to(_TORCH_DEVICE)

    EPOCHS = 12
    t0 = time.time()
    for epoch in range(EPOCHS):
        mlp.train()
        epoch_loss = 0.0
        for xb, yb in loader:
            xb, yb = xb.to(_TORCH_DEVICE), yb.to(_TORCH_DEVICE)
            optimizer.zero_grad()
            out = mlp(xb)
            loss = criterion(out, yb)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * xb.size(0)
        epoch_loss /= len(loader.dataset)
        if (epoch + 1) % 3 == 0 or epoch == 0:
            print(f"  epoch {epoch+1:>2}/{EPOCHS} — loss {epoch_loss:.4f}")
    elapsed = time.time() - t0

    mlp.eval()
    with torch.no_grad():
        val_logits = mlp(X_val_t)
        val_proba_mlp = torch.softmax(val_logits, dim=1).cpu().numpy()
        val_pred_mlp = val_proba_mlp.argmax(axis=1)

    mlp_val_f1 = f1_score(Y_Validation_aligned, val_pred_mlp, average='macro')
    proba_val['MLP (TF-IDF)'] = val_proba_mlp
    results['MLP (TF-IDF)'] = {'model': None, 'train_macro': None, 'val_macro': mlp_val_f1, 'time': elapsed}
    print(f"\n✓ MLP (TF-IDF) — Val Macro F1 = {mlp_val_f1:.4f}  ({elapsed:.1f}s on {_TORCH_DEVICE})")

    mlp_state = {k: v.cpu() for k, v in mlp.state_dict().items()}
    mlp_arch = {'input_dim': input_dim, 'n_classes': n_classes, 'hidden': (256, 128)}
else:
    mlp_state, mlp_arch = None, None
    print("⚠ PyTorch unavailable — MLP skipped, Voting will use the remaining models only.")

## <span style="color:#fff;">Leaderboard</span>
<div style="height:3px;width:48px;background:#2563eb;border-radius:2px;margin:2px 0 10px;"></div>



In [ ]:
leaderboard = pd.DataFrame([
    {'model': k, 'val_macro_f1': v['val_macro'], 'train_macro_f1': v['train_macro'], 'time_s': v['time']}
    for k, v in results.items()
]).sort_values('val_macro_f1', ascending=False).reset_index(drop=True)

print(leaderboard.to_string())
print(f"\nBest single model: {leaderboard.iloc[0]['model']}  (Val Macro F1 = {leaderboard.iloc[0]['val_macro_f1']:.4f})")

## <span style="color:#fff;">Final Model: Soft-Voting Ensemble</span>
<div style="height:3px;width:48px;background:#2563eb;border-radius:2px;margin:2px 0 10px;"></div>

Averages predicted-class probabilities across every model above (classical + MLP) and takes the argmax — the final, required ensembling step.

In [ ]:
# Final step: soft-Voting ensemble, weighted by validation Macro F1.
# Implemented as a manual probability average (rather than sklearn's
# VotingClassifier) so cuML, sklearn, and the PyTorch MLP can all take part
# uniformly -- cuML/torch objects aren't always full sklearn-clonable
# estimators, which VotingClassifier requires.
#
# Equal-weight averaging let very weak models (Naive Bayes ~0.26, KNN ~0.47
# in the first run) drag a ~0.72-caliber ensemble down toward their own
# level. Weighting each model's vote by its own validation Macro F1 -- and
# dropping anything that barely beats random guessing (1/7 for 7 classes)
# -- fixes that without hardcoding specific model names, so it stays
# correct even if a different model turns out to be the weak one.
CLASSES_SORTED = sorted(np.unique(Y_Train_aligned))

MIN_VAL_F1 = 0.55
weights = {name: results[name]['val_macro'] for name in proba_val if results[name]['val_macro'] >= MIN_VAL_F1}
dropped = [name for name in proba_val if name not in weights]
if dropped:
    print(f"Excluded from the vote (val Macro F1 < {MIN_VAL_F1}): {dropped}")

total_w = sum(weights.values())
avg_proba = sum(proba_val[name] * (w / total_w) for name, w in weights.items())
voting_pred = np.array(CLASSES_SORTED)[avg_proba.argmax(axis=1)]

voting_val_f1 = f1_score(Y_Validation_aligned, voting_pred, average='macro')
print(f"\n✓ Weighted soft-Voting ensemble ({len(weights)} of {len(proba_val)} models) — Val Macro F1 = {voting_val_f1:.4f}")
print(f"  vs. best single model                                         — Val Macro F1 = {leaderboard.iloc[0]['val_macro_f1']:.4f}")

final_pred = voting_pred
final_macro_f1 = voting_val_f1
final_micro_f1 = f1_score(Y_Validation_aligned, voting_pred, average='micro')

print("\nConfusion Matrix — Voting ensemble:")
cm = confusion_matrix(Y_Validation_aligned, voting_pred)
print(cm)

## <span style="color:#fff;">Save Artifacts</span>
<div style="height:3px;width:48px;background:#2563eb;border-radius:2px;margin:2px 0 10px;"></div>



In [ ]:
sklearn_models = {name: info['model'] for name, info in results.items() if info['model'] is not None}

with open(ARTIFACT_DIR + '03_modeling.pkl', 'wb') as f:
    pickle.dump({
        'sklearn_models': sklearn_models,
        'mlp_state': mlp_state,
        'mlp_arch': mlp_arch,
        'model_names': list(proba_val.keys()),
        'weights': weights,
        'final_pred': final_pred,
        'final_macro_f1': final_macro_f1,
        'final_micro_f1': final_micro_f1,
        'leaderboard': leaderboard,
        'cm': cm,
        'evaluation_final': evaluation_final,
        'evaluation_text_svd': evaluation_text_svd,
        'evaluation': evaluation,
        'submission': submission,
        'BASE_PATH': BASE_PATH,
        'SEED': SEED,
        'gpu_backend': _GPU_BACKEND,
    }, f)

print(f"✓ Saved modeling artifacts to {ARTIFACT_DIR}03_modeling.pkl")